<a href="https://colab.research.google.com/github/yetinam/pyocto/blob/examples/02_velocity_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![image](https://raw.githubusercontent.com/yetinam/pyocto/main/docs/_static/pyocto_logo_outlined.svg)

*This code is necessary on colab to install PyOcto. If PyOcto is already installed on your machine, you can skip this. In addition, we install PyArrow and Pyrocko. PyArrow is required for reading our example data in parquet format. PyRocko for creating the 1D velocity models.*

In [ ]:
!pip install pyocto pyarrow pyrocko

# Velocity models

This tutorial introduces the PyOcto velocity models. PyOcto supports homogeneous models and 1D layered velocity models. For this tutorial, we assume familiarity with the basics of PyOcto. Take a look at the basics tutorial if you haven't done so yet.

In [ ]:
import pyocto
import pandas as pd
import datetime
import matplotlib.pyplot as plt
import numpy as np

For this notebook, we use the same example data as for the basics notebook, i.e., synthetic events in Chile.

In [ ]:
!wget https://github.com/yetinam/pyocto/raw/main/tests/data/picks
!wget https://github.com/yetinam/pyocto/raw/main/tests/data/stations
!wget https://github.com/yetinam/pyocto/raw/main/tests/data/graeber.csv

In [ ]:
picks = pd.read_parquet("picks")
picks["time"] = picks["time"].apply(lambda x: x.timestamp())
stations = pd.read_parquet("stations")

In [ ]:
stations

## Homogeneous velocity models

In the basics tutorial, we already introduced the homogeneous velocity model. It assumes a constant P and S velocity everywhere in the medium. This assumption is usually sufficient for shallow seismicity and even for subduction zones leads to good results. Let's create a velocity model.

In [ ]:
velocity_model = pyocto.VelocityModel0D(
    p_velocity=7.0,
    s_velocity=4.0,
    tolerance=2.0,
)

With the velocity model, we create an associator instance and run the association. This will take a few seconds.

In [ ]:
associator = pyocto.OctoAssociator.from_area(
    lat=(-25, -18),
    lon=(-71.5, -68),
    zlim=(0, 200),
    time_before=300,
    velocity_model=velocity_model,
    n_picks=10,
    n_p_and_s_picks=4,
)
associator.transform_stations(stations)
events_0d, assignments_0d = associator.associate(picks, stations)

## 1D velocity models

Using a 1D velocity model consists of two steps. First, we need to prepare the model. Second, we create an instance of the model to pass to PyOcto. Let's start by loading our velocity model from a csv file.

In [ ]:
layers = pd.read_csv("graeber.csv")
layers

For creating a velocity model, we need a data frame with three columns (and all other columns will be ignored): depth, vp, vs. The depth describes the depth in kilometers and vp and vs the P and S wave velocity at these depth. Velocities are interpolated linearly between the provided depth levels.

To prepare the velocity model, we use the `create_model` function. This function will create a travel time grid and save it in a binary format. It takes five arguments:
- The velocity model in the data frame
- The spacing between grid nodes in km
- The horizontal extent of the grid
- The vertical extent of the grid
- The path to write the final model to

The velocity grid will always be anchored at depth 0 and distance 0.

In [ ]:
model_path = "velocity_model"
pyocto.VelocityModel1D.create_model(layers, 1., 400, 250, model_path)

To load the model, we use a similar call as for the homogeneous model. However, instead of passing p and s velocities, we now pass the model path. Note that this means that we can reuse a model that we created once multiple times.

In [ ]:
velocity_model = pyocto.VelocityModel1D(model_path, tolerance=2.0)

In [ ]:
associator = pyocto.OctoAssociator.from_area(
    lat=(-25, -18),
    lon=(-71.5, -68),
    zlim=(0, 200),
    time_before=300,
    velocity_model=velocity_model,
    n_picks=10,
    n_p_and_s_picks=4,
)
associator.transform_stations(stations)
events_1d, assignments_1d = associator.associate(picks, stations)

Let's compare the outputs of the two models. In this case, both models found the same number of events (often the 1D model finds more). However, the 1D model was able to associate substantially more picks to these events.

In [ ]:
print("0D model - Events: ", len(events_0d), "Picks: ", events_0d["picks"].sum())
print("1D model - Events: ", len(events_1d), "Picks: ", events_1d["picks"].sum())

Let's compare the detected event catalogs visually. The two catalogs show both systematic and stoachstic differences. The homogeneous model consistently overestimates depth of the deeper events as the assumed phase velocities here are too low. In addition, the model shows more scatter as the locations are not as focussed.

In [ ]:
fig = plt.figure(figsize=(9, 8))
axs = fig.subplots(1, 2, sharex=True, sharey=True)

for ax in axs:
    ax.set_aspect("equal")
    ax.set_xlabel("Easting [km]")
    ax.set_ylabel("Northing [km]")

axs[0].set_title("Homogeneous model")
axs[1].set_title("1D layered model")
axs[0].scatter(events_0d["x"], events_0d["y"], c=events_0d["z"])
axs[1].scatter(events_1d["x"], events_1d["y"], c=events_1d["z"])

## Station specific 1D velocity models

VelocityModel1D corrects station elevations by assuming a constant velocity between the earth surface and the sea level and a vertical incidence. The approximation is problematic for shallow events with similar depths to stations (e.g. borehole stations, OBS). In addition, events above the sea level are not allowed.

For events with similar depths to stations, we can use `StationSpecificVelocityModel1D` to model station elevation more precisely. It calculates different travel time tables for different stations and doesn't require an approximate elevation correction. `StationSpecificVelocityModel1D` consumes more memory and running time than `VelocityModel1D`.

Note that the purpose is to deal with station elevation properly and the subsurface velocity model remains the same for each station.

We use the velocity model loaded before. The format is the same as the one required by VelocityModel1D.

In [ ]:
layers[["depth","vp","vs"]]

We use the `create_model` to generate the travel time tables. It takes two more arguments than `VelocityModel1D.create_model()`:
- **station**: a data frame with two columns: id, elevation. The id column denotes the station identifiers, and the elevation column describes the station elevations in meters above sea level. Any other columns will be ignored. Note that in the later step of association, the same station id should be used in the data frame of stations passed to the `OctoAssociator.associate` function.
- **z_padding_thickness**: thickness of the padding layers above the stations.

In addition, the **path** argument points to a directory in which travel time tables for differnt stations are write. The other arguments are the same as those for `VelocityModel1D.create_model()`.

The velocity grid is anchored at depth *- station elevation - z_padding_thickness* and distance 0.

In [ ]:
model_dir_path = "station_specific_velocity_model"
pyocto.StationSpecificVelocityModel1D.create_model(
    model=layers,
    delta=1.0,
    xdist=400,
    zdist=250,
    z_padding_thickness=5,
    station=stations[["id","elevation"]],
    path=model_dir_path,
)

In the output directory, each travel time table file is named as **id.pyocto**.

In [ ]:
! ls "station_specific_velocity_model"

Besides the travel time tables, there is a binary file named **n_padding**, which stores the number of padding layers `n_padding = int(np.ceil(z_padding_thickness / delta))`

In [ ]:
import struct
with open("station_specific_velocity_model/n_padding", "rb") as f:
    content=f.read()
print(struct.unpack("i",content))

To load the model, we use a similar call as for the 1D model.

In [ ]:
velocity_model = pyocto.StationSpecificVelocityModel1D(model_dir_path, tolerance=2.0)

In [ ]:
associator = pyocto.OctoAssociator.from_area(
    lat=(-25, -18),
    lon=(-71.5, -68),
    zlim=(0, 200), # z can be negative
    time_before=300,
    velocity_model=velocity_model,
    n_picks=10,
    n_p_and_s_picks=4,
)
associator.transform_stations(stations)
events_ss_1d, assignments_ss_1d = associator.associate(picks, stations)

Let's compare the outputs of the three models. The same number of events can be found. The station specific 1D model can associate the most picks and has the smallest residuals for both P and S.

In [ ]:
for mod_type, events, assignments in zip(
    ["0D model", "1D model", "Station specific 1D model"],
    [events_0d, events_1d, events_ss_1d],
    [assignments_0d, assignments_1d, assignments_ss_1d]
):
    num_ev=len(events)
    num_pick=events['picks'].sum()
    mean_p_res=np.mean(assignments[assignments["phase"] == "P"]["residual"])
    mean_s_res=np.mean(assignments[assignments["phase"] == "S"]["residual"])
    print(f"{mod_type:>25s} - Events:{num_ev:3d} | Picks:{num_pick:5d} | Mean P residual:{mean_p_res:7.4f} | Mean S residual: {mean_s_res:7.4f}")

Let's look at the event locations. The 1D model and the station specific 1D model show similar event distribution.

In [ ]:
fig,axs = plt.subplots(2, 3,figsize=(9, 9), sharex=True, sharey="row",height_ratios=[7,1])


    
axs[0][0].set_ylabel("Northing [km]")
axs[1][0].set_ylabel("Depth [km]")

axs[0][0].set_title("Homogeneous model")
axs[0][1].set_title("1D layered model")
axs[0][2].set_title("Station specific 1D model")

axs[0][0].scatter(stations["x"], stations["y"], marker="v", color="red",label="Stations",s=10)
axs[0][0].scatter(events_0d["x"], events_0d["y"], c=events_0d["z"],label="Events",s=10)
axs[1][0].scatter(events_0d["x"], events_0d["z"], c=events_0d["z"],label="Events",s=10)

axs[0][1].scatter(stations["x"], stations["y"], marker="v", color="red",s=10)
axs[0][1].scatter(events_1d["x"], events_1d["y"], c=events_1d["z"],s=10)
axs[1][1].scatter(events_1d["x"], events_1d["z"], c=events_1d["z"],s=10)

axs[0][2].scatter(stations["x"], stations["y"], marker="v", color="red",s=10)
axs[0][2].scatter(events_ss_1d["x"], events_ss_1d["y"], c=events_ss_1d["z"],s=10)
axs[1][2].scatter(events_ss_1d["x"], events_ss_1d["z"], c=events_ss_1d["z"],s=10)
axs[0][0].legend(loc="lower right")

for ax in axs[1,:]:
    ax.set_xlabel("Easting [km]")
    ax.invert_yaxis()

for ax in axs.flat:
    ax.set_xlim(min(np.min(events_0d["x"]),np.min(events_1d["x"]),np.min(events_ss_1d["x"])),max(np.max(events_0d["x"]),np.max(events_1d["x"]),np.max(events_ss_1d["x"])))

## Additional configuration options

Velocity models offer additional configuration options. We already used the `tolerance` parameter. This parameter describes by how much an observed and a predicted travel time are allowed to differ. This accounts for inaccuracies in the velocity model and the pick times. In general, a lower tolerance will lead to fewer detected events, a higher tolerance to more events. However, a too high tolerance will lead to too many false detections. In addition, higher tolerance values lead to longer run times.

On important option for large deployments is the `association_cutoff_distance`. For large deployments, it is often sufficient to only search among stations up to a certain distance, if their picks could be associated. Afterwards, picks at further away stations can be added in the localisation and pick matching stage. This safes substantial runtime. Let's try this out with a 1D velocity model.

In [ ]:
velocity_model = pyocto.VelocityModel1D(model_path, tolerance=2.0, association_cutoff_distance=250)
associator = pyocto.OctoAssociator.from_area(
    lat=(-25, -18),
    lon=(-71.5, -68),
    zlim=(0, 200),
    time_before=300,
    velocity_model=velocity_model,
    n_picks=10,
    n_p_and_s_picks=4,
)
associator.transform_stations(stations)
events, assignments = associator.associate(picks, stations)

Comparing the outputs, we found the same number of events, even though we lost 10 picks through the cutoff. If these picks were actual picks or false associations is unclear.

In [ ]:
print("1D model (250 km cutoff) - Events: ", len(events), "Picks: ", events["picks"].sum())
print("1D model - Events: ", len(events_1d), "Picks: ", events_1d["picks"].sum())

If you want to know more details on velocity models and their parameters, check out the [documentation page](https://pyocto.readthedocs.io/en/latest/pages/velocity_models.html).